<a class="anchor" id="0"></a>
# **Mubileum Duration Generation with VAE**

<a class="anchor" id="0.1"></a>
# **Table of Contents**

- 1. [Setup](#1)
- 2. [Data Preparation](#2)
    - 2.1 [Data Preparation for LSTM](#2.1)
- 3. [Model architecture](#3)
- 4. [Model Tranning](#3)
- 5. [Generate examples](#3)

# **1. SetUp** <a class="anchor" id="1"></a>

[Table of Contents](#0.1)

In [1]:
# Data Manipulation and Analysis
import numpy as np
import pandas as pd

# Scikit-Learn for Machine Learning
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# TensorFlow and Keras for Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Lambda, RepeatVector
from tensorflow.keras import backend as K

# **2. Data Preparation** <a class="anchor" id="2"></a>

[Table of Contents](#0.1)

In [2]:
# Read a CSV file into a Pandas DataFrame
df = pd.read_csv("/kaggle/input/mobileum/jul_aug_sep_oct_2022_msc_1.csv", sep=";")

In [3]:
# columns_cat = ['Call Type', 'Event Type', 'Subscriber Type', 'Traffic Type', 'Service Type', 'Partner Description']
# columns_num = ['Start Hour', 'Event Duration']

# Filter the DataFrame based on specific conditions
df_fil = df[(df['Call Type'] == 'VOICE') & (df['Event Type'] == 'ORIGINATED') & (df['Subscriber Type'] == 'Postpaid')].copy()

# Convert 'Start Date' to datetime
df_fil['TIMESTAMP'] = pd.to_datetime(df_fil['Start Date'] + ' ' + df_fil['Start Hour'].astype(str), format='%d/%m/%Y %H')

# Drop unnecessary columns
df_fil.drop(columns=['Datasource', 'Call Type', 'Event Type', 'Subscriber Type', '#Events', 'Partner Description', 'Start Date', 'Start Hour'], inplace=True)

# Group by 'TIMESTAMP' and sum 'Event Duration'
df_fil = df_fil.groupby(['TIMESTAMP'], as_index=False)['Event Duration'].sum()

# sets the "TIMESTAMP" column as the index of the DataFrame
df_fil.set_index("TIMESTAMP", inplace=True)

# **2.1 Data Preparation for LSTM** <a class="anchor" id="2.1"></a>

[Table of Contents](#0.1)

In [4]:
data = df_fil['Event Duration'].values

# Create sliding windows
window_size = 24
step_size = 1

X = []
y = []

for i in range(0, len(data) - window_size, step_size):
    X.append(data[i:i+window_size])
    y.append(data[i+window_size])

# Convert to numpy arrays
X = np.array(X)
y = np.array(y)

# Reshape X to fit the LSTM input requirement [samples, time steps, features]
X = X.reshape((X.shape[0], window_size, 1))

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

(X_train.shape, X_test.shape), (y_train.shape, y_test.shape)


(((1668, 24, 1), (418, 24, 1)), ((1668,), (418,)))

# **3. Model Architecture** <a class="anchor" id="3"></a>

[Table of Contents](#0.1)

In [5]:
# Define the VAE model architecture
lstm_units = 12
input_size = 24  # As defined by your sliding window size
latent_dim = 2   # Dimensionality of the latent space

# Encoder
encoder_inputs = Input(shape=(input_size, 1))
encoder_lstm, state_h, state_c = LSTM(lstm_units, return_state=True)(encoder_inputs)
z_mean = Dense(latent_dim)(state_h)
z_log_var = Dense(latent_dim)(state_h)

# Sampling function
def sampling(args):
    z_mean, z_log_var = args
    batch = K.shape(z_mean)[0]
    epsilon = K.random_normal(shape=(batch, latent_dim))
    return z_mean + K.exp(0.5 * z_log_var) * epsilon

# Use the Lambda layer to wrap the sampling function
z = Lambda(sampling, output_shape=(latent_dim,))([z_mean, z_log_var])

# Define the decoder model
decoder_input = Input(shape=(latent_dim,))
decoder_state_input_h = Input(shape=(lstm_units,))
decoder_state_input_c = Input(shape=(lstm_units,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_lstm = LSTM(lstm_units, return_sequences=True, stateful=False)
decoder_lstm_out = decoder_lstm(RepeatVector(input_size)(decoder_input), initial_state=decoder_states_inputs)
decoder_model = Model([decoder_input] + decoder_states_inputs, decoder_lstm_out)

# Decoder
decoder_outputs = decoder_lstm(RepeatVector(input_size)(z))

# VAE Model
vae = Model(encoder_inputs, decoder_outputs)

# VAE Loss Function
def vae_loss(encoder_inputs, decoder_outputs):
    reconstruction_loss = K.mean(K.square(encoder_inputs - decoder_outputs))
    kl_loss = -0.5 * K.mean(1 + z_log_var - K.square(z_mean) - K.exp(z_log_var), axis=-1)
    return reconstruction_loss + kl_loss

# Add the loss function to the VAE model
vae.add_loss(vae_loss(encoder_inputs, decoder_outputs))
vae.compile(optimizer='adam')

# **4. Model Trainning** <a class="anchor" id="4"></a>

[Table of Contents](#0.1)

In [6]:

batch_size = 32
epochs = 50

history = vae.fit(X_train, shuffle=True, epochs=epochs, batch_size=batch_size, validation_data=(X_test, None))

# Function to generate new sequences
def generate_sequence(latent_dim, decoder_model, lstm_units):
    # Generate a random latent space vector
    random_latent_vector = np.random.normal(size=(1, latent_dim))
    # Generate the initial states of the LSTM
    initial_h = np.zeros((1, lstm_units))
    initial_c = np.zeros((1, lstm_units))
    return decoder_model.predict([random_latent_vector, initial_h, initial_c])


Epoch 1/50
53/53 [==============================] - 7s 42ms/step - loss: 57298319835136.0000 - val_loss: 56241153900544.0000
Epoch 2/50
53/53 [==============================] - 1s 24ms/step - loss: 57298307252224.0000 - val_loss: 56241141317632.0000
Epoch 3/50
53/53 [==============================] - 1s 24ms/step - loss: 57298319835136.0000 - val_loss: 56241132929024.0000
Epoch 4/50
53/53 [==============================] - 1s 24ms/step - loss: 57298307252224.0000 - val_loss: 56241132929024.0000
Epoch 5/50
53/53 [==============================] - 1s 25ms/step - loss: 57298319835136.0000 - val_loss: 56241132929024.0000
Epoch 6/50
53/53 [==============================] - 2s 29ms/step - loss: 57298315640832.0000 - val_loss: 56241132929024.0000
Epoch 7/50
53/53 [==============================] - 1s 24ms/step - loss: 57298298863616.0000 - val_loss: 56241132929024.0000
Epoch 8/50
53/53 [==============================] - 1s 23ms/step - loss: 57298303057920.0000 - val_loss: 56241132929024.0000


# **5. Generating new examples** <a class="anchor" id="5"></a>

[Table of Contents](#0.1)

In [7]:
new_sequences = generate_sequence(latent_dim, decoder_model, lstm_units)

1/1 [==============================] - 0s 466ms/step


In [8]:
new_sequences.shape

(1, 24, 12)

In [9]:
new_sequences

array([[[ 0.03501599,  0.01893021, -0.00573142,  0.01207284,
          0.01830749,  0.03930858,  0.03026779,  0.03807192,
          0.06816282,  0.00370692, -0.01355153,  0.0257194 ],
        [ 0.08806588,  0.0542862 ,  0.00857203,  0.04233957,
          0.05715306,  0.09905109,  0.07392517,  0.09392069,
          0.14908333,  0.02698447, -0.00464941,  0.06391217],
        [ 0.18922864,  0.13037859,  0.06027251,  0.11134047,
          0.14840534,  0.21155697,  0.15531711,  0.19524667,
          0.2668679 ,  0.08926577,  0.0452068 ,  0.14383501],
        [ 0.4088284 ,  0.31877738,  0.21436583,  0.28306487,
          0.37047046,  0.44366467,  0.34147903,  0.40504548,
          0.46844962,  0.25882956,  0.20977648,  0.3432713 ],
        [ 0.76626027,  0.6983979 ,  0.5949448 ,  0.6505478 ,
          0.74922144,  0.79015404,  0.70639426,  0.74954355,
          0.76755434,  0.6464191 ,  0.62024903,  0.728767  ],
        [ 0.9606474 ,  0.94682187,  0.91613996,  0.93064064,
          0.9603741